# Indicadores de seguimiento del PNIE <br> Parte IV: BI, IOARR y vinculaciones

Autor: Paolo Gutierrez Chochoca

Correo: paolo1309z@gmail.com

Fecha de creación: 16/09/2025

Fecha de actualización: //2025

## 01. Configuración

In [2]:
# Importando librerias
import os
import numpy as np
import pandas as pd
from dbfread import DBF
import re
import unicodedata

In [3]:
# Directorio de trabajo
file1 = 'E:/OneDrive - Ministerio de Educación/00. Paolo/Actividades'
file1 = 'D:/OneDrive - Ministerio de Educación/00. Paolo/Actividades'
file2 = '01. Evaluación PNIE/02. IS-PNIE'
file = file1 + '/' + file2
os.chdir(file)

In [4]:
# Configurando el formato
pd.options.mode.chained_assignment = None
pd.options.display.float_format = '{:,.1f}'.format

In [5]:
# Definiendo funciones de conteo
def unique(data, varlist):
    aux = data[varlist]
    a = aux.drop_duplicates().shape[0]
    list = " ".join(varlist)
    print(f'Number of unique values of {list} is {a:,.0f}')
    print(f'Number of records is {data.shape[0]:,.0f}')

def by_unique(data, varlist, var):
    unique(data, varlist)
    list = varlist + var
    aux = data[list]
    a = aux.drop_duplicates().value_counts(var).reset_index().\
    sort_values(by=var)
    print("\n", a)

In [6]:
# Corrección de codigos locales
def codlocal(df, oldvar, newvar):
    df[newvar] = df[oldvar].astype(str).str.zfill(6)

## 02. Vinculaciones CUI-codlocal

### 2.1 Vinculaciones de la UPI

In [7]:
file = 'data/UPI/3. vinculación entre CUI y código modular.xlsx'
df_0 = pd.read_excel(file, sheet_name='vinculaciones')

In [8]:
df = df_0.copy()

# Seleccionando variables
df = df[['Código Inversión', 'Código Local']]
df = df.loc[~(df['Código Local'].isna())]
df['Código Local'] = pd.to_numeric(df['Código Local'], 
                                   errors='coerce').astype('Int64')
codlocal(df, 'Código Local', 'codlocal')
df.drop(columns='Código Local', inplace=True)
df.rename({'Código Inversión': 'CUI'}, axis=1, inplace=True)
df['CUI'] = df['CUI'].astype('str')
df.drop_duplicates(inplace=True)

upi = df.copy()
upi.head()

,CUI,codlocal
0,2698362,429209
1,2698293,244673
2,2698291,515141
3,2698285,622184
4,2698261,788565


### 2.2 Vinculaciones del MEF

In [9]:
file = 'data/HR 170461-2025 - Proyectos de inversion.xlsx'
df_0 = pd.read_excel(file, sheet_name='inv')

In [10]:
df = df_0.copy()

# Seleccionando variables
df = df[['CODIGO_UNICO', 'CODIGO_SNIP', 'CODIGO_MODULAR_IIEE', 'CODIGO_LOCAL']]
df.loc[df['CODIGO_UNICO'].isna(), 'CODIGO_UNICO'] = \
    df.loc[df['CODIGO_UNICO'].isna(), 'CODIGO_SNIP']
df['CODIGO_UNICO'] = pd.to_numeric(df['CODIGO_UNICO'], errors='coerce'). \
    astype('Int64')
df.rename({'CODIGO_UNICO': 'CUI'}, axis=1, inplace=True)
df['CUI'] = df['CUI'].astype('str')

df = df.loc[~(df['CODIGO_MODULAR_IIEE'].isna() | df['CODIGO_LOCAL'].isna())]

df['CODIGO_MODULAR_IIEE'] = pd.to_numeric(df['CODIGO_MODULAR_IIEE'], 
                                          errors='coerce').astype('Int64')
df.rename({'CODIGO_MODULAR_IIEE': 'codmod'}, axis=1, inplace=True)
df['codmod'] = df['codmod'].astype(str).str.zfill(7)

df['CODIGO_LOCAL'] = pd.to_numeric(df['CODIGO_LOCAL'], errors='coerce'). \
    astype('Int64')
codlocal(df, 'CODIGO_LOCAL', 'codlocal')
df = df[['CUI', 'codlocal']]

# Eliminando vinculaciones no escolarizadas
df = df.loc[df['codlocal'] != '000000']
df.drop_duplicates(inplace=True)

mef = df.copy()

### 2.3 Consolidación de vinculaciones

In [11]:
vin = pd.merge(upi, mef,
               on=['CUI', 'codlocal'],
               how='outer',
               indicator=True)

vin['fuente'] = 'UPI y MEF'
vin.loc[vin['_merge'] == 'left_only', 'fuente'] = 'UPI'
vin.loc[vin['_merge'] == 'right_only', 'fuente'] = 'MEF'
print(vin['fuente'].value_counts(dropna=False))

# Exportando data
vin = vin[['CUI', 'codlocal', 'fuente']]
vin.drop_duplicates(inplace=True)
vin.to_csv('data/procesadas/vinculaciones.csv', index=False)

fuente
UPI          145547
UPI y MEF     49637
MEF            2981
Name: count, dtype: int64


## 03. Banco de inversiones y criterios de focalización

### 3.1 Criterios de focalización de las inversiones:
1. La inversión debe estar vinculada al menos a un local educativo.
2. La inversión no debe estar desactivada.
3. La inversión debe tener un avance físico mayor o igual al 85%.
4. La inversión debe tener un devengado acumulado mayor o igual al 85% de su costo actualizado.

In [14]:
file = 'data/UPI/2025.08.25 Base de Inversiones.xlsx'
bi_0 = pd.read_excel(file, sheet_name='Data')

In [15]:
bi = bi_0.copy()

# Selecccionando información
bi.columns = bi.iloc[3]
bi = bi.iloc[4:-1, :].reset_index(drop=True)

# 1. La inversión debe estar vinculada al menos a un local educativo
bi = pd.merge(bi, vin['CUI'],
              left_on='CODIGO_INVERSION',
              right_on='CUI',
              how='left',
              indicator=True)
bi['cf1'] = (bi['_merge'] == 'both').astype('int8')
print(bi['cf1'].value_counts(dropna=False), '\n')
bi.drop_duplicates(inplace=True)

# 2. La inversión no debe estar desactivada
print(bi['ESTADO'].value_counts(dropna=False), '\n')
desac = ['DESACTIVADO PERMANENTE', 'DESACTIVADO TEMPORAL', 'DESACTIVADO', 
         'DESACTIVADA', np.nan]
bi['cf2'] = (~(bi['ESTADO'].isin(desac))).astype('int8')
print(bi['cf2'].value_counts(dropna=False), '\n')

# 3. La inversión debe tener un avance físico mayor o igual al 85%
bi['AVANCE_FISICO_F9'] = pd.to_numeric(bi['AVANCE_FISICO_F9'])
bi['AVANCE_FISICO_F12B'] = pd.to_numeric(bi['AVANCE_FISICO_F12B'])
bi['cf3'] = 0
bi.loc[bi['AVANCE_FISICO_F12B'] >= 0.85, 'cf3'] = 1
bi.loc[(bi['AVANCE_FISICO_F12B'] >= 0.85) & (bi['cf3'] == 0), 'cf3'] = 1
print(bi['cf3'].value_counts(dropna=False), '\n')

# 4. La inversión debe tener un devengado acumulado mayor o igual al 85% de su 
# costo actualizado
bi['P_EJEC_COSTO_TOTAL'] = pd.to_numeric(bi['P_EJEC_COSTO_TOTAL'])
bi['cf4'] = 0
bi.loc[bi['P_EJEC_COSTO_TOTAL'] >= 0.85, 'cf4'] = 1
print(bi['cf4'].value_counts(dropna=False), '\n')

# Inversiones que cumplen con los criterios de focalización
mask = (bi['cf1'] == 1) & (bi['cf2'] == 1) & (bi['cf3'] == 1) & \
    (bi['cf4'] == 1)
bi['cf'] = mask.astype('int8')
print(bi['cf'].value_counts(dropna=False), '\n')

cf1
1    198059
0     99698
Name: count, dtype: int64 

ESTADO
ACTIVO                    56115
DESACTIVADO PERMANENTE    43872
CERRADO                   21081
ACTIVA                     9786
NaN                        8022
DESACTIVADA                4475
DESACTIVADO TEMPORAL        798
DESACTIVADO                 133
PREINVERSIÓN                  1
LIQUIDACIÓN DE OBRAS          1
Name: count, dtype: int64 

cf2
1    86984
0    57300
Name: count, dtype: int64 

cf3
0    133313
1     10971
Name: count, dtype: int64 

cf4
0    109087
1     35197
Name: count, dtype: int64 

cf
0    139042
1      5242
Name: count, dtype: int64 



### 3.2 Año de culminación

F9 > Último devengado > F12B

In [16]:
# F9
bi['fecha_F9'] = pd.to_datetime(bi['FEC_REG_CIERRE_F9'], errors='coerce')
bi['anio_F9'] = bi['fecha_F9'].dt.year.astype('Int64', errors='ignore')

# F12B
bi['fecha_F12'] = pd.to_datetime(bi['FECHA_ULT_ACT_F12B'], errors='coerce')
bi['anio_F12'] = bi['fecha_F12'].dt.year.astype('Int64', errors='ignore')

# Año de culminación
bi['anio_culm'] = pd.Series([pd.NA] * len(bi), dtype="Int64")
bi.loc[(~(bi['anio_F9'].isna())) & (bi['cf'] == 1) & (bi['anio_culm'].isna()), 
       'anio_culm'] = bi['anio_F9']
bi.loc[(~(bi['ANO_ULTIMO_DEV'].isna())) & (bi['cf'] == 1) & \
       (bi['anio_culm'].isna()), 'anio_culm'] = bi['ANO_ULTIMO_DEV']
bi.loc[(~(bi['anio_F12'].isna())) & (bi['cf'] == 1) & (bi['anio_culm'].isna()), 
       'anio_culm'] = bi['anio_F12']
bi.loc[bi['cf'] == 1, 'anio_culm'].value_counts(dropna=False)

# Exportando la data
bi.drop_duplicates(inplace=True)
bi.to_csv('data/procesadas/banco_inversiones_criterios.csv', index=False)

### 3.3 Tipo de gobierno

In [17]:
bi.loc[bi['cf'] == 1, 'NIVEL'].value_counts(dropna=False)

NIVEL
GL    3436
GR    1564
GN     242
Name: count, dtype: int64

In [23]:
bi.loc[(bi['ALTERNATIVA'].isna()) & (bi['cf'] == 1), 'DES_TIPO_FORMATO'].value_counts(dropna=False)

DES_TIPO_FORMATO
IOARR                    2163
PROYECTO EMBLEMATICO       91
PROYECTO DE INVERSION      26
Name: count, dtype: int64

In [20]:
bi['ALTERNATIVA']

0         INSTALACIÓN DE UN MINICOMPLEJO DEPORTIVO QUE C...
1         ·\tConstrucción De 04 Aula de Concreto Armado....
2         Construcción de infraestructura tipo aporticad...
3         ACTIVIDADES DE CAPACITACIÓN SOCIAL, A LOS MIEM...
4         Construcción de pabellon, adquisición de muebl...
                                ...                        
297752                                                  NaN
297753                                                  NaN
297754                                                  NaN
297755                                                  NaN
297756                                                  NaN
Name: ALTERNATIVA, Length: 144284, dtype: object

#### 3.3.1 GR y GL

In [82]:
bi_sub = bi.loc[(bi['cf'] == 1) & (bi['NIVEL'] != 'GN')]
col = ['CODIGO_INVERSION', 
       'NOMBRE_INVERSION', 'DES_TIPO_FORMATO', 'TIPO_IOARR', 'ESTADO',
       'SITUACION', 'ALTERNATIVA', 'NIVEL', 'PLIEGO',
       'COD_EJECUTORA', 'UEP_ULTIMA', 'COD_SEC_EJEC_UEP_ULT', 'UEP_PRINCIPAL',
       'COD_SEC_EJEC_UEP_PRINC', 'UF', 'UEI', 'OPMI',
       'FUNCION', 'PROGRAMA', 'SUB_PROGRAMA',
       'FECHA_REGISTRO', 'FECHA_VIABILIDAD',
       'COSTO_ACTUALIZADO_BI', 'COSTO_INV_TOTAL_BI', 'COSTO_INV_TOTAL_PMI',
       'TIENE_ET_DE', 'FECHA_ET_DE', 'ULT_MODIF_ET_DE',
       'TIENE_F8', 'ETAPA_F8', 'ULT_SEC_F8', 'ULT_MODIF_F8', 'FEC_INI_F8',
       'FEC_FIN_F8', 'COMPONENTES_F8', 'COMPONENTES_DEV_SIAF_HISTORICO',
       'TIENE_F12B', 'FECHA_ULT_ACT_F12B', 'ULT_FEC_DECLA_ESTIM_F12B',
       'DES_SITUACION_F12B', 'FEC_SITUACION_F12B', 'AVANCE_EJECUCION_F12B',
       'AVANCE_FISICO_F12B',
       'TIENE_F9', 'CERRADO_F9', 'CULMINADA_F9', 'AVANCE_FISICO_F9',
       'DES_CIERRE_F9', 'FEC_REG_CIERRE_F9',
       'CARTERA_PMI', 'CICLO_INVERSION_PMI', 'ORDEN_PRELACION_PMI',
       'PROXY_PRELACION', 'anio_culm']
bi_sub = bi_sub[col]
bi_sub.rename({'CODIGO_INVERSION': 'CUI'}, axis=1, inplace=True)

# Exportando la data
bi_sub.drop_duplicates(inplace=True)
bi_sub.to_csv('data/procesadas/banco_inversiones_gr_gl.csv', index=False)

#### 3.3.2 GN

In [83]:
bi_nac = bi.loc[(bi['cf'] == 1) & (bi['NIVEL'] == 'GN')]
col = ['CODIGO_INVERSION', 
       'NOMBRE_INVERSION', 'DES_TIPO_FORMATO', 'TIPO_IOARR', 'ESTADO',
       'SITUACION', 'ALTERNATIVA', 'NIVEL', 'PLIEGO',
       'COD_EJECUTORA', 'UEP_ULTIMA', 'COD_SEC_EJEC_UEP_ULT', 'UEP_PRINCIPAL',
       'COD_SEC_EJEC_UEP_PRINC', 'UF', 'UEI', 'OPMI',
       'FUNCION', 'PROGRAMA', 'SUB_PROGRAMA',
       'FECHA_REGISTRO', 'FECHA_VIABILIDAD',
       'COSTO_ACTUALIZADO_BI', 'COSTO_INV_TOTAL_BI', 'COSTO_INV_TOTAL_PMI',
       'TIENE_ET_DE', 'FECHA_ET_DE', 'ULT_MODIF_ET_DE',
       'TIENE_F8', 'ETAPA_F8', 'ULT_SEC_F8', 'ULT_MODIF_F8', 'FEC_INI_F8',
       'FEC_FIN_F8', 'COMPONENTES_F8', 'COMPONENTES_DEV_SIAF_HISTORICO',
       'TIENE_F12B', 'FECHA_ULT_ACT_F12B', 'ULT_FEC_DECLA_ESTIM_F12B',
       'DES_SITUACION_F12B', 'FEC_SITUACION_F12B', 'AVANCE_EJECUCION_F12B',
       'AVANCE_FISICO_F12B',
       'TIENE_F9', 'CERRADO_F9', 'CULMINADA_F9', 'AVANCE_FISICO_F9',
       'DES_CIERRE_F9', 'FEC_REG_CIERRE_F9',
       'CARTERA_PMI', 'CICLO_INVERSION_PMI', 'ORDEN_PRELACION_PMI',
       'PROXY_PRELACION', 'anio_culm']
bi_nac = bi_nac[col]
bi_nac.rename({'CODIGO_INVERSION': 'CUI'}, axis=1, inplace=True)

# Exportando la data
bi_nac.drop_duplicates(inplace=True)
bi_nac.to_csv('data/procesadas/banco_inversiones_gn.csv', index=False)

## 04. IOARR

### 4.1 Procesamiento de la información

In [36]:
file = 'data/UPI/REP_INV_MINEDU_F8_IOARR.xlsx'
ioarr_0 = pd.read_excel(file, sheet_name='INVERSIONES')

In [37]:
ioarr = ioarr_0.copy()

# Selecccionando información
ioarr = ioarr_0.iloc[:, 1:].reset_index(drop=True)
ioarr['CODIGO_UNICO'] = ioarr['CODIGO_UNICO'].astype('str')

# Seleccionando variables
ioarr = ioarr[['CODIGO_UNICO', 'DES_TIPO_INVERSION', 'DES_NATURALEZA',
               'DES_TIPO_COMPONENTE', 'DES_ACTIVO']]

# Obteniendo información del BI
col = ['CODIGO_INVERSION', 
       'NOMBRE_INVERSION', 'DES_TIPO_FORMATO', 'TIPO_IOARR', 'ESTADO',
       'SITUACION', 'ALTERNATIVA', 'NIVEL', 'PLIEGO',
       'COD_EJECUTORA', 'UEP_ULTIMA', 'COD_SEC_EJEC_UEP_ULT', 'UEP_PRINCIPAL',
       'COD_SEC_EJEC_UEP_PRINC', 'UF', 'UEI', 'OPMI',
       'FUNCION', 'PROGRAMA', 'SUB_PROGRAMA',
       'FECHA_REGISTRO', 'FECHA_VIABILIDAD',
       'COSTO_ACTUALIZADO_BI', 'COSTO_INV_TOTAL_BI', 'COSTO_INV_TOTAL_PMI',
       'TIENE_ET_DE', 'FECHA_ET_DE', 'ULT_MODIF_ET_DE',
       'TIENE_F8', 'ETAPA_F8', 'ULT_SEC_F8', 'ULT_MODIF_F8', 'FEC_INI_F8',
       'FEC_FIN_F8', 'COMPONENTES_F8',
       'TIENE_F12B', 'FECHA_ULT_ACT_F12B', 'ULT_FEC_DECLA_ESTIM_F12B',
       'DES_SITUACION_F12B', 'FEC_SITUACION_F12B', 'AVANCE_EJECUCION_F12B',
       'AVANCE_FISICO_F12B',
       'TIENE_F9', 'CERRADO_F9', 'CULMINADA_F9', 'AVANCE_FISICO_F9',
       'DES_CIERRE_F9', 'FEC_REG_CIERRE_F9',
       'CARTERA_PMI', 'CICLO_INVERSION_PMI', 'ORDEN_PRELACION_PMI',
       'PROXY_PRELACION', 'cf', 'anio_culm']
ioarr = pd.merge(ioarr, bi[col],
                 left_on='CODIGO_UNICO', right_on='CODIGO_INVERSION',
                 how='left')

# Seleccionando IOARR que cumplen con os criterios de focalización
ioarr = ioarr.loc[ioarr['cf'] == 1]
ioarr.drop_duplicates(inplace=True)

### 4.2 Identificando vinculo con el indicador

In [38]:
ioarr['ind'] = ''

print(ioarr['DES_TIPO_COMPONENTE'].value_counts(dropna=False))

DES_TIPO_COMPONENTE
INFRAESTRUCTURA            3177
EQUIPAMIENTO                699
MOBILIARIO                  280
TERRENOS                     36
INFRAESTRUCTURA NATURAL      17
INTANGIBLES                   9
VEHICULOS                     3
Name: count, dtype: int64


#### 4.2.1 Vehiculos

In [39]:
print(ioarr.loc[ioarr['DES_TIPO_COMPONENTE'] == 'VEHICULOS', 
          'DES_ACTIVO'].unique())
ioarr.loc[ioarr['DES_TIPO_COMPONENTE'] == 'VEHICULOS', 
          'ind'] = 'No corresponde'

['VEHICULO' 'VEHÍCULOS DE SERVICIOS DE TRANSPORTE' 'MOTONAVE FLUVIAL']


#### 4.2.2 Intangibles

In [41]:
print(ioarr.loc[ioarr['DES_TIPO_COMPONENTE'] == 'INTANGIBLES', 
          'DES_ACTIVO'].unique())
ioarr.loc[ioarr['DES_TIPO_COMPONENTE'] == 'INTANGIBLES', 
          'ind'] = 'No corresponde'

['SOFTWARE PARA LA GESTION' 'TALLER' 'CAPACIDAD ORGANIZACIONAL'
 'SOFTWARE DE VENTAS Y MERCADEO' 'POZO DE EXTRACCION' 'SOFTWARE'
 'AMBIENTE DE PREPARACION Y EXPENDIO DE ALIMENTOS' 'CAPACIDAD DIGITAL'
 'SOFTWARE DE ADMINISTRACION']


#### 4.2.3 Infraestructura natural

In [42]:
print(ioarr.loc[ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA NATURAL', 
                'DES_ACTIVO'].unique(), '\n')

act = ['RED DE ALCANTARILLADO', 
       'INSTALACIONES EXTERIORES DE SERVICIOS BASICOS']
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA NATURAL') & 
          (ioarr['DES_ACTIVO'].isin(act)), 'ind'] = 'GI2.1'

act = ['AULA DE EDUCACION PRIMARIA', 'AULA DE EDUCACION SECUNDARIA',
       'AMBIENTE ADMINISTRATIVO', 'AMBIENTE COMPLEMENTARIO']
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA NATURAL') & 
          (ioarr['DES_ACTIVO'].isin(act)) & 
          (ioarr['DES_TIPO_INVERSION'] == 'AMPLIACION MARGINAL DE SERVICIOS'), 
          'ind'] = 'GI4.4'
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA NATURAL') & 
          (ioarr['DES_ACTIVO'].isin(act)) & 
          (ioarr['DES_TIPO_INVERSION'] == 'OPTIMIZACIÓN'), 
          'ind'] = 'GI4.1'

act = ['SERVICIOS HIGIENICOS Y/O VESTIDORES']
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA NATURAL') & 
          (ioarr['DES_ACTIVO'].isin(act)), 'ind'] = 'GI2.2'

act = ['LAVADERO MULTIUSO']
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA NATURAL') & 
          (ioarr['DES_ACTIVO'].isin(act)), 'ind'] = 'GI2.2'

ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA NATURAL') & 
          (ioarr['ind'] == ''), 'ind'] = 'No corresponde'

print(ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA NATURAL') &
                (ioarr['ind'] == ''), 
                'DES_ACTIVO'].unique())

['RED DE ALCANTARILLADO' 'LAVADERO MULTIUSO'
 'SERVICIOS HIGIENICOS Y/O VESTIDORES' 'COBERTURA'
 'AULA DE EDUCACION PRIMARIA' 'ESPACIO DEPORTIVO ABIERTO' 'HUERTO'
 'CAMPO DEPORTIVO' 'INSTALACIONES EXTERIORES DE SERVICIOS BASICOS'
 'AMBIENTE COMPLEMENTARIO' 'AULA DE EDUCACION SECUNDARIA'
 'COBERTURA DE INSTALACIONES DEPORTIVAS' 'AMBIENTE ADMINISTRATIVO'] 

[]


#### 4.2.4 Terrenos

In [43]:
print(ioarr.loc[ioarr['DES_TIPO_COMPONENTE'] == 'TERRENOS', 
                'DES_ACTIVO'].unique(), '\n')

act = ['UNIDAD BASICA DE SANEAMIENTO (UBS)']
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'TERRENOS') & 
          (ioarr['DES_ACTIVO'].isin(act)), 'ind'] = 'GI2.1'

act = ['TERRENO']
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'TERRENOS') & 
          (ioarr['DES_ACTIVO'].isin(act)) &
          (ioarr['DES_TIPO_INVERSION'] == \
           'AMPLIACIÓN MARGINAL - ADQUISICION ANTICIPADA DE TERRENOS'), 
           'ind'] = 'GI2.1'

ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'TERRENOS') & 
          (ioarr['ind'] == ''), 'ind'] = 'No corresponde'

['UNIDAD BASICA DE SANEAMIENTO (UBS)' 'TERRENO'
 'TERRENO PARA RESIDENCIA UNIVERSITARIA'] 



#### 4.2.5 Mobiliario y equipamiento

In [44]:
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'MOBILIARIO') | 
          (ioarr['DES_TIPO_COMPONENTE'] == 'EQUIPAMIENTO'), 'ind'] = 'GI3.2'

#### 4.2.6 Infraestructura

In [45]:
# Reforzamiento estructural
act = ['AULA DE INNOVACION PEDAGOGICA', 'MURO DE CONTENCION',
       'EDIFICACION', 'AULA DE EDUCACION PRIMARIA', 
       'AULA DE EDUCACION SECUNDARIA', 'AMBIENTE COMPLEMENTARIO', 'AULA']
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA') & 
          (ioarr['DES_ACTIVO'].isin(act)) &
          (ioarr['DES_NATURALEZA'] == 'REFORZAMIENTO ESTRUCTURAL'), 
          'ind'] = 'GI1.2'
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA') & 
          (ioarr['ind'] == '') &
          (ioarr['DES_NATURALEZA'] == 'REFORZAMIENTO ESTRUCTURAL'), 
          'ind'] = 'No corresponde'

# Remoción
act = ['BLOQUE DE INFRAESTRUCTURA']
ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA') & 
          (ioarr['DES_ACTIVO'].isin(act)) &
          (ioarr['DES_NATURALEZA'] == 'REMOCION'), 
          'ind'] = 'No corresponde'



ioarr.loc[(ioarr['DES_TIPO_COMPONENTE'] == 'INFRAESTRUCTURA') & 
          (ioarr['ind'] == '') &
          (ioarr['DES_NATURALEZA'] == 'REPARACION'), 
          'DES_TIPO_INVERSION'].unique()

array(['REHABILITACIÓN', 'AMPLIACION MARGINAL DE SERVICIOS'], dtype=object)

In [51]:
ioarr.loc[ioarr['ind'] == ''].to_excel('data/procesadas/ioarr_pendiente.xlsx', 
                                     index=False)

In [50]:
ioarr.columns

Index(['CODIGO_UNICO', 'DES_TIPO_INVERSION', 'DES_NATURALEZA',
       'DES_TIPO_COMPONENTE', 'DES_ACTIVO', 'CODIGO_INVERSION',
       'NOMBRE_INVERSION', 'DES_TIPO_FORMATO', 'TIPO_IOARR', 'ESTADO',
       'SITUACION', 'ALTERNATIVA', 'NIVEL', 'PLIEGO', 'COD_EJECUTORA',
       'UEP_ULTIMA', 'COD_SEC_EJEC_UEP_ULT', 'UEP_PRINCIPAL',
       'COD_SEC_EJEC_UEP_PRINC', 'UF', 'UEI', 'OPMI', 'FUNCION', 'PROGRAMA',
       'SUB_PROGRAMA', 'FECHA_REGISTRO', 'FECHA_VIABILIDAD',
       'COSTO_ACTUALIZADO_BI', 'COSTO_INV_TOTAL_BI', 'COSTO_INV_TOTAL_PMI',
       'TIENE_ET_DE', 'FECHA_ET_DE', 'ULT_MODIF_ET_DE', 'TIENE_F8', 'ETAPA_F8',
       'ULT_SEC_F8', 'ULT_MODIF_F8', 'FEC_INI_F8', 'FEC_FIN_F8',
       'COMPONENTES_F8', 'TIENE_F12B', 'FECHA_ULT_ACT_F12B',
       'ULT_FEC_DECLA_ESTIM_F12B', 'DES_SITUACION_F12B', 'FEC_SITUACION_F12B',
       'AVANCE_EJECUCION_F12B', 'AVANCE_FISICO_F12B', 'TIENE_F9', 'CERRADO_F9',
       'CULMINADA_F9', 'AVANCE_FISICO_F9', 'DES_CIERRE_F9',
       'FEC_REG_CIERRE_F9', 'C

## 05. Proyectos de inversión y otros - GR/GL

In [84]:
file = 'data/procesadas/banco_inversiones_gr_gl.csv'
df_0 = pd.read_csv(file, dtype={'CUI': str})
df_0['DES_TIPO_FORMATO'].value_counts(dropna=False)

DES_TIPO_FORMATO
PROYECTO DE INVERSION    7031
IOARR                    2575
FUR                        61
PROYECTO EMBLEMATICO        1
Name: count, dtype: int64

### 5.1 FUR y proyecto emblemático

In [98]:
df = df_0.copy()

# No existe descripción detallada de la intervención
tipos = ['FUR', 'PROYECTO EMBLEMATICO']
df.loc[df['DES_TIPO_FORMATO'].isin(tipos), 'ALTERNATIVA'].unique()

# Supuesto: Estos proyectos implican la sustitución total de la infraestructura
df['sust_total'] = (df['DES_TIPO_FORMATO'].isin(tipos)).astype('Int8')
df['sust_total'].value_counts(dropna=False)

sust_total
0    9606
1      62
Name: count, dtype: Int64

### 5.2 Proyecto de inversión

In [303]:
pi = df.loc[df['DES_TIPO_FORMATO'] == 'PROYECTO DE INVERSION']

#### 5.2.1 Grupo de intervención 1

##### GI1.1

In [304]:
grupos = [
    ['demolicion', 'demoler'],
    ['provisional', 'provisionales', 'prefabricada', 'prefabricadas',
     'modular', 'modulares', 'temporal', 'temporales']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI1_1_pre',
    whole_word=True
)

freq(pi, 'GI1_1_pre')

            Freq   (%)
GI1_1_pre             
0          6,970  99.1
1             61   0.9
Total      7,031 100.0 



##### GI1.2

In [305]:
grupos = [
    ['reforzamiento', 'reforzar'],
    ['estructural', 'estructura']
]

#prohibidas = ['demolicion', 'construccion', 'construir']

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
#    forbidden=prohibidas,
    new_col='GI1_2_pre',
    whole_word=True
)

freq(pi, 'GI1_2_pre')

            Freq   (%)
GI1_2_pre             
0          7,010  99.7
1             21   0.3
Total      7,031 100.0 



##### GI1.3

In [306]:
grupos = [
    ['intervencion'],
    ['contingente'],
    ['geotextil']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI1_3_pre',
    whole_word=True
)

freq(pi, 'GI1_3_pre')

            Freq   (%)
GI1_3_pre             
0          7,031 100.0
Total      7,031 100.0 



##### GI1.4

In [307]:
grupos = [
    ['modular', 'modulares'],
    ['selva', 'amazonia', 'amazonico', 'tropical']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI1_4_pre',
    whole_word=True
)

freq(pi, 'GI1_4_pre')

            Freq   (%)
GI1_4_pre             
0          7,031 100.0
Total      7,031 100.0 



##### GI1.5

In [308]:
grupos = [
    ['cerco', 'muro'],
    ['perimetrico']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI1_5_pre',
    whole_word=True
)

freq(pi, 'GI1_5_pre')

            Freq   (%)
GI1_5_pre             
0          3,968  56.4
1          3,063  43.6
Total      7,031 100.0 



#### 5.2.2 Grupo de intervención 2

##### GI2.1

In [309]:
grupos = [
    ['agua', 'potable', 'desague', 'desagüe', 'alcantarillado', 'saneamiento']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI2_1_pre',
    whole_word=True
)

freq(pi, 'GI2_1_pre')

            Freq   (%)
GI2_1_pre             
0          6,512  92.6
1            519   7.4
Total      7,031 100.0 



##### GI2.2

In [310]:
grupos = [
    ['almacenamiento', 'cisterna', 'tanque', 'elevado', 'bombeo', 'bomba',
     'higienicos', 'ss.hh.', 'sshh', 'inodoro', 'urinario', 'lavadero',
     'bebedero', 'bebederos', 'drenaje', 'lluvia', 'lluvia', 'pluvial', 
     'canaleta', 'canaletas']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI2_2_pre',
    whole_word=True
)

freq(pi, 'GI2_2_pre')

            Freq   (%)
GI2_2_pre             
0          3,959  56.3
1          3,072  43.7
Total      7,031 100.0 



#### 5.2.3 Grupo de intervención 3

##### GI3.1

In [311]:
grupos = [
    ['cable', 'cableado', 'tablero', 'tableros', 'gabinete', 'gabinetes',
     'interruptores', 'interruptor', 'puesta a tierra', 
     'instalaciones internas']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI3_1_pre',
    whole_word=True
)

freq(pi, 'GI3_1_pre')

            Freq   (%)
GI3_1_pre             
0          6,771  96.3
1            260   3.7
Total      7,031 100.0 



##### GI3.2

In [312]:
grupos = [
    ['mobiliario', 'muebles', 'carpeta', 'silla', 'escritorio', 'pizarra',
     'mesa', 'sillon', 'corral'], 
    ['equipamiento', 'equipos']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI3_2_pre',
    whole_word=False
)

freq(pi, 'GI3_2_pre')

            Freq   (%)
GI3_2_pre             
0          3,893  55.4
1          3,138  44.6
Total      7,031 100.0 



#### 5.2.4 Grupo de intervención 4

##### GI4.1

Supuesto: La sustitución parcial se da mediante IOARRs

##### GI4.2

In [313]:
grupos = [
    ['electrica', 'energia', 'electricidad', 'eolica', 'solar']
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI4_2_pre',
    whole_word=True
)

freq(pi, 'GI4_2_pre')

            Freq   (%)
GI4_2_pre             
0          6,743  95.9
1            288   4.1
Total      7,031 100.0 



##### GI4.3

In [314]:
grupos = [
    ['rampa', 'ascensor', 'elevador', 'inodoro accesible', 'discapacidad', 
     "instalaciones sanitarias accesibles", "aparatos sanitarios", 
     "sshh accesibles", "inodoro accesible", "bano accesible", 
     "bateria sanitaria accesible", "lavatorio accesible", 
     "personas con discapacidad", "discapacidad"]
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI4_3_pre',
    whole_word=True
)

freq(pi, 'GI4_3_pre')

            Freq   (%)
GI4_3_pre             
0          6,828  97.1
1            203   2.9
Total      7,031 100.0 



##### GI4.4

In [315]:
grupos = [
    ["ampliacion", "ampliar", "ampliada", "incremento de area", 
     "nuevos espacios"],
    ["infraestructura", "local educativo", "edificacion educativa", 
     "espacios educativos"]
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='GI4_4_pre',
    whole_word=True
)

freq(pi, 'GI4_4_pre')

            Freq   (%)
GI4_4_pre             
0          6,315  89.8
1            716  10.2
Total      7,031 100.0 



#### 5.2.5 Grupo de intervención 5

In [316]:
grupos = [
    ['sustitucion', 'sustituir', 'construccion'],
    ['edificacion', 'pabellon', 'aula', 'infraestructura'],
]

pi = flag_word_groups_in_columns(
    pi, 
    cols=['NOMBRE_INVERSION', 'ALTERNATIVA'],
    groups=grupos,
    new_col='sust_total_nueva_infra',
    whole_word=False
)

freq(pi, 'sust_total_nueva_infra')

                         Freq   (%)
sust_total_nueva_infra             
1                       4,935  70.2
0                       2,096  29.8
Total                   7,031 100.0 



In [317]:
# Obteniendo información sobre LLEE nuevos y existente
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})
vinc = pd.read_csv('data/procesadas/vinculaciones.csv',
                   dtype={'codlocal': str, 'CUI': str})

aux = pd.merge(pi, vinc[['CUI', 'codlocal']],
               on='CUI', how='left', indicator='CUI_codlocal')
aux = pd.merge(aux, exis,
               on='codlocal', how='left', indicator='existente')

# Elaborando los indicadores del GI5
aux = aux[['CUI', 'sust_total_nueva_infra', 'existente']]
aux.drop_duplicates(inplace=True)

aux['GI5_1_pre'] = 0
aux.loc[(aux['sust_total_nueva_infra'] == 1) & (aux['existente'] == 'both'),
       'GI5_1_pre'] = 1

aux['GI5_2_pre'] = 0
aux.loc[(aux['sust_total_nueva_infra'] == 1) & (aux['existente'] == 'left_only'),
       'GI5_2_pre'] = 1

# Dimensionando a nivel de CUI
aux = aux.groupby('CUI')[['GI5_1_pre', 'GI5_2_pre']].sum().reset_index()
freq(aux, 'GI5_1_pre')
freq(aux, 'GI5_2_pre')

# Consolidando
pi = pd.merge(pi, aux, on='CUI', how='left')

            Freq   (%)
GI5_1_pre             
1          4,612  65.6
0          2,418  34.4
Total      7,030 100.0 

            Freq   (%)
GI5_2_pre             
0          6,635  94.4
1            395   5.6
Total      7,030 100.0 



In [ ]:
# Corrigiendo el CUI 2397607
pi.loc[pi['CUI'] == '2397607', 'CULMINADA_F9'] = 'SÍ'
pi.loc[pi['CUI'] == '2397607', 'AVANCE_FISICO_F9'] = 100
pi.loc[pi['CUI'] == '2397607', 'FEC_REG_CIERRE_F9'] = '2025-08-19'
pi.loc[pi['CUI'] == '2397607', 'anio_culm'] = 2025
pi.drop_duplicates(inplace=True)

In [ ]:
######## PENDIENTE: INCLUIR SUSTITUCION TOTAL DE FUR Y EMBLEMATICOS ############

In [249]:
def freq(df, columna, dropna=False, decimales=2):
    """
    Genera una tabla de frecuencias absolutas, relativas (%) y totales.

    Parámetros:
    -----------
    df : DataFrame
        El DataFrame de entrada
    columna : str
        Nombre de la columna a analizar
    dropna : bool, opcional
        Si True, ignora los valores NaN. Default = False
    decimales : int, opcional
        Número de decimales para el porcentaje. Default = 2
    """
    
    # Frecuencias absolutas y relativas
    freq = df[columna].value_counts(dropna=dropna)
    rel = df[columna].value_counts(normalize=True, dropna=dropna) * 100
    
    # Construcción de la tabla
    tabla = pd.DataFrame({
        'Freq': freq,
        '(%)': rel.round(decimales)
    })
    
    # Totales
    tabla.loc['Total'] = [
        tabla['Freq'].sum(),
        tabla['(%)'].sum().round(decimales)
    ]
    
    # Asegurar que Frecuencia sea entero (excepto el Total que se mantiene correcto)
    tabla['Freq'] = tabla['Freq'].apply(lambda x: f"{int(x):,}" if pd.notna(x) else x)
 
    return print(tabla, '\n')

In [101]:
def _normalize_text(s: pd.Series) -> pd.Series:
    # Quita tildes/diacríticos y pasa a minúsculas
    def _norm(x):
        if x is None:
            return ""
        x = str(x)
        x = unicodedata.normalize("NFKD", x)
        x = "".join(c for c in x if not unicodedata.combining(c))
        return x.lower()
    return s.fillna("").astype(str).map(_norm)

In [102]:
def flag_word_groups_in_columns(
    df: pd.DataFrame,
    cols: list,
    groups: list[list[str]],
    new_col: str = "flag",
    whole_word: bool = True,
    forbidden: list[str] = None
) -> pd.DataFrame:

    if not cols:
        raise ValueError("Debes proporcionar al menos una columna en `cols`.")
    for c in cols:
        if c not in df.columns:
            raise KeyError(f"Columna no encontrada: {c}")

    # normaliza texto de columnas
    proc_cols = [_normalize_text(df[c]) for c in cols]
    text = proc_cols[0]
    for s in proc_cols[1:]:
        text = text.str.cat(s, sep=" ")

    # === 1. Palabras requeridas (AND entre grupos, OR dentro del grupo) ===
    group_masks = []
    for group in groups:
        words = ["".join(
            ch for ch in unicodedata.normalize("NFKD", str(w).lower())
            if not unicodedata.combining(ch)
        ) for w in group if str(w).strip()]
        if not words:
            # grupo vacío -> no aporta; marca como True para no afectar el AND
            group_masks.append(pd.Series(True, index=df.index))
            continue
        inner = "|".join(re.escape(w) for w in words)
        pat = rf"(?u)\b(?:{inner})\b" if whole_word else inner
        group_masks.append(text.str.contains(pat, regex=True, na=False))

    mask = np.logical_and.reduce(group_masks) if group_masks else \
        pd.Series(False, index=df.index)
    
    # === 2. Palabras prohibidas ===
    if forbidden:
        forbidden_words = ["".join(
            ch for ch in unicodedata.normalize("NFKD", str(w).lower())
            if not unicodedata.combining(ch)
        ) for w in forbidden if str(w).strip()]
        inner_forbidden = "|".join(re.escape(w) for w in forbidden_words)
        pat_forbidden = rf"(?u)\b(?:{inner_forbidden})\b" if whole_word \
            else inner_forbidden
        mask_forbidden = text.str.contains(pat_forbidden, regex=True, na=False)

        # si aparece alguna palabra prohibida, fuerza a 0
        mask = mask & ~mask_forbidden

    df[new_col] = mask.astype("uint8")
    return df